# Module 1: Build a Deep Agent

> Self-directed module, ~45 min.

Deep Agents = `create_agent()` + a pre-built middleware stack (filesystem, planning, subagents, context management). We build up from a bare agent to a fully-featured client research assistant, exploring:

- The harness and built-in tools
- Custom tools (web search, client profile lookup)
- Subagents and context isolation
- Backends and persistent memory
- Middleware (compliance, audit)
- Human-in-the-loop on tool calls
- AGENTS.md and Skills


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model

from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from langsmith import Client, uuid7
from IPython.display import Image, display

from utils.config import PROJECT_NAME

# Print the project URL so the "Verify in LangSmith" cells below have a direct deep link.
_client = Client()
try:
    _project = _client.read_project(project_name=PROJECT_NAME)
    print(f"Project URL: {_project.url}")
except Exception:
    print(f"Project '{PROJECT_NAME}' will appear in LangSmith once the first invoke runs.")

print("Ready")


---
# Part 1: Deep Agents

Deep Agents = `create_agent()` + a pre-built middleware stack (filesystem, planning, subagents, context management).

We build up from a bare agent to a fully-featured client research assistant.


## 1.1 Your First Deep Agent

`create_deep_agent()` gives you a filesystem, a todo list, and context management out of the box — no tools required.

<img src="../images/deepAgentsDiag.png" style="width: auto; max-height: 420px; border-radius: 8px;">

### What you get for free

- **Filesystem Tools** — `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`
- **Planning Tool** — `write_todos` for task tracking
- **Subagent Delegation** — `task()` tool for isolated work
- **Large Tool Result Eviction** — Automatically offloads tool results >20k tokens to the filesystem
- **Conversation Summarization** — Compresses history when approaching ~85% context capacity
- **Dangling Tool Call Patching** — Fixes message history consistency automatically


In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    system_prompt="You are a helpful assistant.",
    checkpointer=MemorySaver(),
)
agent


In [ ]:
# The agent can already write and read files — these are built-in tools
config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Write a haiku about AI to /haiku.txt, then read it back to me."}]
}, config=config)

print(result["messages"][-1].content)


### Verify in LangSmith

Click the **Project URL** printed in the Setup cell above and find this run in the **Traces** tab. The trace tree shows the bare deep agent's built-in tools (`write_file`, `read_file`) being invoked even though no custom tools were registered — they come with the harness.


In [ ]:
# Helper: print the virtual filesystem from a deep agent result.
def print_files(result, header="VIRTUAL FILESYSTEM (in-memory, not on disk!)"):
    files = result.get("files") or {}
    if not files:
        print("(no files in state)")
        return
    print("=" * 50)
    print(header)
    print("=" * 50)
    for path, file_data in files.items():
        print(f"\n  Path: {path!r}")
        print("  " + "-" * 38)
        content = file_data
        if isinstance(file_data, dict) and "content" in file_data:
            content = file_data["content"]
        if isinstance(content, list):
            content = "\n".join(content)
        for line in str(content).splitlines():
            print(f"  | {line}")

print_files(result)


### Filesystem persistence within a thread

By default, `create_deep_agent()` uses **StateBackend** — files are stored in agent state and persist within a thread (via the checkpointer), but disappear when you start a new thread.

| Backend | Storage | Persistence | Use Case |
|---------|---------|-------------|----------|
| **StateBackend** | In-memory (agent state) | Single thread | Scratch pads, intermediate results |
| **FilesystemBackend** | Local disk | Permanent | Direct file access (use with caution) |
| **StoreBackend** | LangGraph Store | Cross-thread | Long-term memories |
| **CompositeBackend** | Routes to others | Mixed | Selective persistence |


In [ ]:
# Same thread — the file persists via the checkpointer
result = agent.invoke({
    "messages": [{"role": "user", "content": "Read the file /haiku.txt"}]
}, config=config)

print("Same thread:", result["messages"][-1].content)


In [ ]:
# New thread — StateBackend is ephemeral, so the file is gone
new_config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "List all files with ls /"}]
}, config=new_config)

print("New thread:", result["messages"][-1].content)


### Key Takeaway
- `create_deep_agent()` gives you filesystem + planning capabilities for free
- Files are stored in agent state (virtual, not on disk)
- `StateBackend` (default) persists within a thread but is ephemeral across threads
- We'll see how to make files persist across threads with `CompositeBackend` + `StoreBackend` in section 1.4


## 1.2 Custom Tools

Add your own tools alongside the built-in ones. We define `web_search` inline with `@tool` so the pattern stays visible — the body delegates to `resilient_tavily_search` from `utils/search.py`, which retries on Tavily failure and falls back to a canned response so the demo doesn't break on a flaky network.


In [ ]:
from langchain_core.tools import tool
from utils.search import resilient_tavily_search


@tool(parse_docstring=True)
def web_search(query: str) -> str:
    """Search the web for company news, market data, or general information.

    Args:
        query: Search query to execute.
    """
    # Retries on Tavily failure, then falls back to a canned response. See utils/search.py.
    return resilient_tavily_search(query, max_retries=2)


agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt="You are a client research assistant for a wealth management firm.",
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Summarize recent news on Apple (AAPL). Write a one-paragraph summary to /summary.md"}]
}, config=config)

print("Agent reply:", result["messages"][-1].content)
print()
print_files(result)


## 1.3 Subagents: Isolated Delegation

Subagents run in a separate context. The main agent delegates via `task()` and only sees the final result — keeping the main context clean.

<img src="../images/deepAgentSubagents.png" style="width: auto; max-height: 380px; border-radius: 8px;">

For the rest of the notebook we also use a `get_client_profile` lookup tool defined the same way in `agents/client_research_agent.py`. Import it here — `web_search` from §1.2 is still in scope.


In [ ]:
from agents.client_research_agent import get_client_profile
from datetime import datetime

research_subagent = {
    "name": "research-agent",
    "description": "Delegate company / market research tasks. Give one topic at a time.",
    "system_prompt": (
        f"You are a research assistant for a wealth management firm. "
        f"Today is {datetime.now().strftime('%Y-%m-%d')}.\n"
        "Use web_search to gather company news, market data, or general info. "
        "Limit to 3 search calls."
    ),
    "tools": [web_search],
}

agent = create_deep_agent(
    model=model,
    tools=[web_search, get_client_profile],
    system_prompt=(
        "You are a client research assistant for a wealth management firm. "
        "Help relationship managers prep for client meetings by looking up "
        "client portfolios and researching their holdings or relevant market "
        "events. Delegate deep research topics to the research-agent."
    ),
    subagents=[research_subagent],
    checkpointer=MemorySaver(),
)
agent


In [ ]:
config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Lightly research recent news on Apple Inc."}]
}, config=config)

print(result["messages"][-1].content[:1500])


## 1.4 Backends & Memory

By default, files live in ephemeral state (`StateBackend`). Use `CompositeBackend` to route paths — e.g. `/memories/` to persistent `StoreBackend` while everything else stays ephemeral.

<img src="../images/deepAgentBackends.png" style="width: auto; max-height: 380px; border-radius: 8px;">


In [ ]:
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# Pass a CompositeBackend *instance* (not a factory). The namespace stays a
# callable so StoreBackend can read the current Runtime when scoping memory.
backend = CompositeBackend(
    default=StateBackend(),                                  # ephemeral scratch space
    routes={
        "/memories/": StoreBackend(                          # persists across threads
            store=store,
            namespace=lambda rt: ("memories", "shared"),
        ),
    },
)

agent = create_deep_agent(
    model=model,
    tools=[web_search, get_client_profile],
    system_prompt=(
        "You are a client research assistant. Save important client facts to /memories/ for future reference. "
        "ALWAYS check /memories files before answering any questions to ensure you don't miss relevant context."
    ),
    subagents=[research_subagent],
    backend=backend,
    store=store,
    checkpointer=MemorySaver(),
)

# Thread 1: agent saves a memory
config1 = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Remember that Smith Family Office's risk profile is aggressive growth. Save this to /memories/clients/smith-family-office.md"}]
}, config=config1)
print("Thread 1:", result["messages"][-1].content)


In [ ]:
# Thread 2: different thread, but /memories/ persists via StoreBackend
config2 = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "What's Smith Family Office's risk profile? Check /memories/"}]
}, config=config2)
print("Thread 2:", result["messages"][-1].content)


## 1.5 Middleware: Pluggable Behavior

Middleware hooks into `wrap_model_call` (every LLM call) and `wrap_tool_call` (every tool call). This lets you inject rules, audit, or intercept without changing agent code.

<img src="../images/deepAgentMiddleware.png" style="width: auto; max-height: 380px; border-radius: 8px;">

### Built-in context management

Three strategies the deep-agent middleware uses to keep within the model's context window:

<img src="../images/Offloading Inputs LangChain.png" style="width: auto; max-height: 240px; border-radius: 8px;">

**Offload Large Inputs** — file write/edit tool calls leave the full content in conversation history. At ~85% context capacity, deep agents truncate older tool calls and replace them with a file-pointer reference.

<img src="../images/Offloading Results LangChain.png" style="width: auto; max-height: 240px; border-radius: 8px;">

**Offload Large Results** — tool results over ~20k tokens are written to the backend and swapped with a path + 10-line preview. The agent can re-read or grep the full content as needed.

<img src="../images/LangChain Summarization.png" style="width: auto; max-height: 240px; border-radius: 8px;">

**Conversation Summarization** — when there's nothing left to offload and context hits ~85% of `max_input_tokens`, history is summarized. Full messages move to `/conversation_history/`; a structured summary replaces them in working memory.

The cell below shows the **custom-middleware pattern** with two functions of our own — compliance rules and audit logging. This is a general example of how to wire your own middleware into `create_deep_agent`. The three built-in middlewares shown in the diagrams above (input offloading, result offloading, conversation summarization) are automatic and don't require explicit code — they're already in the deep-agent harness.


In [ ]:
from langchain.agents.middleware import wrap_model_call, wrap_tool_call
from langchain_core.messages import SystemMessage

audit_log = []

@wrap_model_call
def compliance_rules(request, handler):
    """Inject compliance rules into every LLM call."""
    rules = """## Compliance Rules
- Never include SSNs, credit card numbers, or account passwords in responses
- Always cite sources when presenting research findings
- Flag any request involving non-public client or financial information"""
    existing = request.system_message
    blocks = list(existing.content_blocks) if existing else []
    blocks.append({"type": "text", "text": f"\n\n{rules}"})
    return handler(request.override(system_message=SystemMessage(content_blocks=blocks)))

@wrap_tool_call
def audit_trail(request, handler):
    """Create an audit log entry for every tool call."""
    entry = {"tool": request.tool_call["name"], "timestamp": datetime.now().isoformat()}
    result = handler(request)
    entry["status"] = "success"
    audit_log.append(entry)
    return result

agent_with_middleware = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt="You are a client research assistant for a wealth management firm.",
    middleware=[compliance_rules, audit_trail],
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_middleware.invoke({
    "messages": [{"role": "user", "content": "Summarize Microsoft's most recent earnings. Write a short summary to /summary.md"}]
}, config=config)

print(result["messages"][-1].content)
print(f"\n--- Audit Log ({len(audit_log)} entries) ---")
for entry in audit_log:
    print(f"  {entry['timestamp']}  {entry['tool']}  {entry['status']}")


## 1.6 HITL: Tool-Level Approval

Deep Agents supports `interrupt_on` — pause execution when specific tools are called. The human can approve, edit, or reject.

<img src="../images/deepAgentHITL.png" style="width: auto; max-height: 380px; border-radius: 8px;">


In [ ]:
agent_with_hitl = create_deep_agent(
    model=model,
    tools=[web_search, get_client_profile],
    system_prompt="You are a client research assistant for a wealth management firm.",
    checkpointer=MemorySaver(),
    interrupt_on={
        "write_file": True,
        "edit_file": True,
    },
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_hitl.invoke({
    "messages": [{"role": "user", "content": "Write a brief on Acme Pension Fund to /acme_brief.md"}]
}, config=config)

if result.get("__interrupt__"):
    interrupt_info = result["__interrupt__"][0].value
    for action in interrupt_info["action_requests"]:
        print(f"Paused — tool: {action['name']}, args: {action['args']}")
    print("\nWaiting for approval...")


In [ ]:
# Approve and continue
if result.get("__interrupt__"):
    result = agent_with_hitl.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,
    )
    print("Approved!")
    print("Agent reply:", result["messages"][-1].content)
    print()
    print_files(result)


## 1.7 AGENTS.md & Skills

`AGENTS.md` replaces hardcoded system prompts with an editable identity file. Skills are loaded on demand — the agent reads them only when the task matches.


In [ ]:
from deepagents.backends.utils import create_file_data

# Inlined here so the notebook is self-contained.
# Source of truth: agents/deployable_agents/client_research/{AGENTS.md, skills/client-brief/SKILL.md}

agents_md = """# Client Research Assistant

You are an expert client research assistant for a wealth management firm. You look up client profiles, research holdings and market events, and produce polished pre-meeting briefs and portfolio updates.

## Workflow
1. Plan with write_todos
2. Look up the client via get_client_profile if a client is named
3. Delegate company / market research to the research-agent via task()
4. Synthesize findings into a brief
5. Save the final brief to /client_brief.md
6. Save key takeaways to /memories/client_notes.md

## Rules
- Always call get_client_profile first when a client is named
- Delegate web research to the research-agent rather than searching directly
- Consolidate citations [1], [2], [3]
- Check /skills/ for content format instructions
"""

client_brief_skill = """---
name: client-brief
description: Write a structured pre-meeting brief for a relationship manager. Use this skill when asked to prep a client meeting, summarize a client's situation, or produce a meeting prep document.
---

# Client Brief Skill

- Header: client name, AUM, last meeting date
- Portfolio Snapshot: top holdings with weights, risk profile
- Holdings Highlights: 2-3 sentences per major holding covering recent news, key drivers, current risks
- Market Context: brief paragraph on broader market themes relevant to the portfolio
- Talking Points: 3-5 bulleted prompts the RM can raise in the meeting
- Sources: numbered list of citations
"""

agent = create_deep_agent(
    model=model,
    tools=[web_search, get_client_profile],
    system_prompt=agents_md,
    subagents=[research_subagent],
    memory=["./AGENTS.md"],
    skills=["./skills/"],
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Prep me for my meeting with Smith Family Office. Write the client brief."}],
    "files": {
        "/AGENTS.md": create_file_data(agents_md),
        "/skills/client-brief/SKILL.md": create_file_data(client_brief_skill),
    },
}, config=config)

print(result["messages"][-1].content[:1500])


### Verify in LangSmith

Open this run via the **Project URL** printed in the Setup cell. In the trace, look for the `read_file` call against `/skills/client-brief/SKILL.md` — that's the skill being loaded on demand once the request matched the skill's description. The `write_file` call for `/client_brief.md` is where the final brief was saved.


## 1.8 The Complete Agent

All pieces together: tools, subagents, memory, middleware, HITL, AGENTS.md, and skills.

> `agents/client_research_agent.py` packages a minimal slice of this (no HITL, no FilesystemBackend, no middleware) for Modules 02–06's evaluations.


In [ ]:
store = InMemoryStore()
audit_log = []  # reset

complete_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        "/memories/": StoreBackend(
            store=store,
            namespace=lambda rt: ("memories", "shared"),
        ),
    },
)

complete_agent = create_deep_agent(
    model=model,
    tools=[web_search, get_client_profile],
    system_prompt=agents_md,
    subagents=[research_subagent],
    backend=complete_backend,
    store=store,
    middleware=[compliance_rules, audit_trail],
    checkpointer=MemorySaver(),
    interrupt_on={"write_file": True, "edit_file": True},
    memory=["./AGENTS.md"],
    skills=["./skills/"],
)

print("Complete agent created with:")
print("  - Custom tools (web_search, get_client_profile)")
print("  - Subagents (research-agent)")
print("  - Memory (/memories/ -> StoreBackend)")
print("  - Middleware (compliance rules + audit trail)")
print("  - HITL (interrupt on file writes)")
print("  - AGENTS.md + Skills")


In [ ]:
# Drive the complete agent end-to-end.
# Exercises: subagent delegation, file writes (HITL-gated),
# /memories/ persistence, and middleware (audit + compliance).
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": str(uuid7())}}

# Seed AGENTS.md + the client-brief skill (same content as the 1.7 cell)
# so the agent has its identity and capabilities loaded.
seed_files = {
    "/AGENTS.md": create_file_data(agents_md),
    "/skills/client-brief/SKILL.md": create_file_data(client_brief_skill),
}

result = complete_agent.invoke({
    "messages": [HumanMessage(content=(
        "Prep me for my meeting with Smith Family Office. "
        "Follow your AGENTS.md workflow: look up the client, delegate research on one major holding, "
        "write the final brief to /client_brief.md, and save key takeaways to /memories/client_notes.md. "
        "Search at most once."
    ))],
    "files": seed_files,
}, config=config)

# The HITL middleware pauses on every write_file / edit_file. Approve them all.
while result.get("__interrupt__"):
    payload = result["__interrupt__"][0].value
    actions = payload.get("action_requests", [])
    for action in actions:
        print(f"  HITL pause -> approving {action['name']}: {action['args'].get('file_path','?')}")
    result = complete_agent.invoke(
        Command(resume={"decisions": [{"type": "approve"} for _ in actions]}),
        config=config,
    )

print("\nFinal reply:\n", result["messages"][-1].content[:600])
print()
# Show only files the agent wrote (skip the seed files we passed in).
seed_paths = set(seed_files.keys())
agent_files = {k: v for k, v in (result.get("files") or {}).items() if k not in seed_paths}
print_files({"files": agent_files}, header="FILES THE AGENT WROTE")

print(f"\nAudit log: {len(audit_log)} tool call(s) recorded by the audit middleware")
for entry in audit_log:
    print(f"  {entry['timestamp']}  {entry['tool']:20s} {entry['status']}")


### Deep Agents Recap

| Feature | How | Built-in? |
|---------|-----|----------|
| **Harness** | `create_deep_agent()` | Filesystem, TodoList, Summarization |
| **Custom tools** | `tools=[web_search, get_client_profile]` | Added to built-in tools |
| **Subagents** | `subagents=[{name, description, ...}]` | `task()` tool |
| **Memory** | `CompositeBackend` routing to `StoreBackend` | Path-based routing |
| **Middleware** | `middleware=[wrap_model_call, wrap_tool_call]` | Appended to built-in stack |
| **HITL** | `interrupt_on={"write_file": True}` | Configurable per tool |
| **AGENTS.md** | `memory=["./AGENTS.md"]` or `files={}` | Editable identity |
| **Skills** | `skills=["./skills/"]` or `files={}` | On-demand capabilities |


## Next Steps

This notebook builds a complete client research agent in-memory. The production-ready variant lives at `agents/deployable_agents/client_research/agent.py` — same shape, except it uses `FilesystemBackend(virtual_mode=True)` instead of `StateBackend` and omits middleware. Modules 02–06 import the eval-safe version from `agents/client_research_agent.py` via `utils.config.load_active_agent()`.

Next: `02_tracing.ipynb`.
